# 03. Bivariate EDA & Relationship Discovery

How to choose the correct visualization and statistical tests for pairs of variables (Num-Num, Cat-Num, Cat-Cat) to guide feature engineering.


## 1. Objective
Learn how to analyze relationships between pairs of variables, detect non-linearity and heteroscedasticity, and choose transformations that linearize complex patterns.


## 2. Dataset & Decision Context
- **Dataset**: Used Car Market (`used_cars.csv`)
- **ML Objective**: Regression to predict `selling_price`
- **Pairs to Explore**:
  1. **Numerical vs Numerical**: `mileage` vs `selling_price`, `engine_cc` vs `selling_price`
  2. **Categorical vs Numerical**: `brand` vs `selling_price`, `fuel_type` vs `selling_price`
  3. **Categorical vs Categorical**: `brand` vs `fuel_type`


## 3. What Should I Check?

| Variable Pair | What to Look For | Downstream Action |
|---|---|---|
| **Num vs Num** | Linearity vs Curvature, Heteroscedasticity | Non-linear $\rightarrow$ Polynomial, Log-Log, or Spline terms |
| **Cat vs Num** | Mean/Median separation, variance equality | High separation $\rightarrow$ Strong candidate for Target Encoding |
| **Cat vs Cat** | Sparsity in contingency tables, structural overlap | Cells with 0 counts $\rightarrow$ Group rare interactions |


## 4. Technique Breakdown

```
WHAT: Bivariate Relationship Analysis (Scatter plots, Log-Log plots, Grouped Boxplots, Crosstabs)
WHY: Identifies non-linear patterns, interaction candidates, and variance expansion
WHEN: Always during feature exploration and feature-target relationship mapping
WHEN NOT: Avoid scatter plots on high-cardinality discrete values with heavy overplotting (use 2D KDE or hexbin)
HOW: Scatter + trendline for Num-Num; Boxplot/Violin for Cat-Num; Crosstab heatmap for Cat-Cat
WHAT TO LOOK FOR: Exponential decay curves, expanding fan shapes (heteroscedasticity)
WHAT ACTION: Apply log transformations to linearize curves; engineer interaction terms
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

df = pd.read_csv('../datasets/used_cars/used_cars.csv')
# Clean known data entry error for exploratory plots
df_clean = df[(df['mileage'] > 0) & (df['engine_cc'] > 0)].copy()
df_clean['vehicle_age'] = 2024 - df_clean['year']
print(f"Cleaned dataset: {df_clean.shape[0]:,} rows")


## 5. Numerical vs Numerical: Detecting Non-Linear Depreciation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Raw Age vs Price (Exponential Decay)
sns.scatterplot(data=df_clean.sample(3000, random_state=42), x='vehicle_age', y='selling_price', 
                alpha=0.3, ax=axes[0], color='#2b5c8f')
sns.regplot(data=df_clean.sample(3000, random_state=42), x='vehicle_age', y='selling_price', 
            scatter=False, ax=axes[0], color='red', line_kws={'linestyle':'--'})
axes[0].set_title('Raw: Vehicle Age vs Selling Price (Non-linear decay)')
axes[0].set_xlabel('Vehicle Age (Years)')
axes[0].set_ylabel('Price ($)')

# 2. Semi-Log: Age vs Log(Price) (Linearized relationship)
df_clean['log_price'] = np.log(df_clean['selling_price'])
sns.scatterplot(data=df_clean.sample(3000, random_state=42), x='vehicle_age', y='log_price', 
                alpha=0.3, ax=axes[1], color='#27ae60')
sns.regplot(data=df_clean.sample(3000, random_state=42), x='vehicle_age', y='log_price', 
            scatter=False, ax=axes[1], color='red')
axes[1].set_title('Semi-Log: Vehicle Age vs Log(Price) (Linearized)')
axes[1].set_xlabel('Vehicle Age (Years)')
axes[1].set_ylabel('log(Price)')

plt.tight_layout()
plt.show()


## 6. Categorical vs Numerical: Brand & Fuel Price Separation


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Brand vs Price (sorted by median price)
brand_order = df_clean.groupby('brand')['selling_price'].median().sort_values(ascending=False).index
sns.boxplot(data=df_clean, x='brand', y='selling_price', order=brand_order, ax=axes[0], color='#2b5c8f')
axes[0].set_title('Selling Price Distribution by Brand (Median Sorted)')
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_yscale('log')
axes[0].set_ylabel('Price ($ - Log Scale)')

# Fuel Type vs Price
fuel_order = df_clean.groupby('fuel_type')['selling_price'].median().sort_values(ascending=False).index
sns.boxplot(data=df_clean, x='fuel_type', y='selling_price', order=fuel_order, ax=axes[1], color='#27ae60')
axes[1].set_title('Selling Price by Fuel Type')
axes[1].set_ylabel('Price ($)')

plt.tight_layout()
plt.show()


## 7. Categorical vs Categorical: Contingency & Interaction Analysis


In [ ]:
ct = pd.crosstab(df_clean['brand'], df_clean['fuel_type'], normalize='index') * 100

plt.figure(figsize=(9, 5))
sns.heatmap(ct, annot=True, fmt='.1f', cmap='Blues', cbar_kws={'label': '% within Brand'})
plt.title('Brand vs Fuel Type Contingency (% Distribution within Brand)')
plt.ylabel('Brand')
plt.xlabel('Fuel Type')
plt.tight_layout()
plt.show()


## 8. Interpretation & Decision Log

### What did we find?
1. **Curvature**: The raw relationship between `vehicle_age` and `selling_price` is strictly exponential ($(1-r)^{age}$). Fitting a linear model on raw features violates the linear assumption. In the semi-log space ($\log(y) \sim \text{age}$), the relationship becomes linear ($R^2$ improves substantially).
2. **Brand Separation**: Luxury brands (Porsche, Mercedes, BMW) maintain distinct valuation baselines compared to economy brands (Hyundai, Nissan).
3. **Contingency Sparsity**: Porsche has almost zero Diesel or Electric listings in this dataset.

### Explicit Decision
> [!IMPORTANT]
> **Decision Rule**:
> - **Because** vehicle price decays exponentially with age, we **will** model the regression target on the log scale $\log(\text{selling\_price})$.
> - **Because** `brand` creates strong median separation across 8 discrete levels, we **will** use One-Hot Encoding for linear models and evaluate target encoding for high-cardinality `model` levels.


## 9. Decision Table: Bivariate Analysis Selection

| Pair Types | Primary Diagnostic Plot | Statistical Test | Target Action |
|---|---|---|---|
| **Continuous vs Continuous** | Scatter + Regplot, Log-Log plot | Pearson $r$, Spearman $\rho$ | Linearization via $\log$ or polynomial terms |
| **Categorical vs Continuous** | Grouped Boxplot, Violin plot | ANOVA $F$-test, Kruskal-Wallis | Feature selection, Target Encoding candidate |
| **Categorical vs Categorical** | Heatmap Crosstab, Stacked Bar | Chi-Square $\chi^2$ Test | Interaction feature creation, rare cell pruning |
